# Customer Churn Prediction — Part 2: Model Building

In this notebook we will:
1. Prepare the data and split it into train / test sets
2. Build a preprocessing **Pipeline** (scaling + encoding)
3. Train 4 baseline models: **Logistic Regression, KNN, Decision Tree, Random Forest**
4. Do **hyperparameter tuning** (`GridSearchCV` and `RandomizedSearchCV`)
5. Compare models, pick the best one, inspect it

## 1. Import libraries

In [20]:
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42

## 2. Load data and select features

In [6]:
df = pd.read_csv("Churn_Modelling.csv")
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
# Identifier columns have no predictive value
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

##### Split input and output feature

In [8]:
X = df.drop(columns="Exited")
y = df["Exited"]

In [9]:
print("Features:", list(X.columns))

Features: ['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']


In [10]:
print("X shape:", X.shape, "& y shape:", y.shape)

X shape: (10000, 10) & y shape: (10000,)


## 3. Train / test split

We use `stratify=y` so that both sets keep the same ~20% churn ratio (because of imbalanced data).

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, "& Test:", X_test.shape)


Train: (8000, 10) & Test: (2000, 10)


## 4. Preprocessing with `ColumnTransformer`

| Columns | Transformation | Why |
|---|---|---|
| `CreditScore, Age, Tenure, Balance, NumOfProducts, EstimatedSalary` | `StandardScaler` | KNN and Logistic Regression are sensitive to feature scale |
| `Geography, Gender` | `OneHotEncoder` | convert text categories to numbers (`handle_unknown='ignore'` so unseen categories don't crash) |
| `HasCrCard, IsActiveMember` | passthrough | already 0/1 |

Putting preprocessing **inside the pipeline** means the scaler/encoder are fitted only on training data (no data leakage, also inside cross-validation) and the same steps are automatically applied to new customers at prediction time.

In [ ]:
preprocessor = ColumnTransformer(transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features))

In [13]:
X

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0.00,1,1,1,101348.88
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58
2,502,France,Female,42,8,159660.80,3,1,0,113931.57
3,699,France,Female,39,1,0.00,2,0,0,93826.63
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10
...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77
9997,709,France,Female,36,7,0.00,1,0,1,42085.58
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52


In [15]:
df.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')

In [14]:
num_features = ["CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"]
cat_features = ["Geography", "Gender"]
bin_features = ["HasCrCard", "IsActiveMember"]

def build_preprocessor():
    return ColumnTransformer(transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
        ("bin", "passthrough", bin_features),
    ])

def build_pipeline(model):
    return Pipeline(steps=[
        ("preprocess", build_preprocessor()),
        ("model", model),
    ])

In [ ]:
prep = build_preprocessor().fit(X_train)
print("Number of features after preprocessing:", len(prep.get_feature_names_out()))
print(list(prep.get_feature_names_out()))

## 5. Baseline models (default hyperparameters)

First we train all four models with their default settings to get a starting point.

Because only ~20% of customers churn, we report **accuracy, precision, recall, F1 and ROC-AUC**:
- **Recall** — of the customers who really left, how many did we catch?
- **Precision** — of the customers we flagged, how many really left?
- **F1** — balance between precision and recall
- **ROC-AUC** — how well the model ranks churners above non-churners

In [16]:
def evaluate(pipe, X_te, y_te):
    pred = pipe.predict(X_te)
    proba = pipe.predict_proba(X_te)[:, 1]
    return {
        "Accuracy": accuracy_score(y_te, pred)
    }

In [17]:
baseline_models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)}

In [19]:
baseline_results = {}
for name, model in baseline_models.items():
    pipe = build_pipeline(model)
    pipe.fit(X_train, y_train)
    scores = evaluate(pipe, X_test, y_test)
    baseline_results[name] = scores

baseline_df = pd.DataFrame(baseline_results).T.round(4)
baseline_df

,Accuracy
Logistic Regression,0.8080
KNN,0.8385
Decision Tree,0.7810


In [ ]:



    "Random Forest": RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
}



**What to notice**
- All models have decent accuracy, but **recall is low** for Logistic Regression and KNN — they miss many churners.
- The default Decision Tree overfits (perfect on train, much weaker on test); Random Forest is already the strongest.
- Next step: tuning should improve these numbers, especially recall/F1.

## 6. Hyperparameter tuning

We tune each pipeline with 5-fold cross-validation on the **training set**, optimising **F1** (a good single metric for imbalanced churn data).

- `GridSearchCV` for Logistic Regression, KNN and Decision Tree (small search spaces → try every combination)
- `RandomizedSearchCV` for Random Forest (large search space → sample 25 random combinations)

Parameters of the final model are addressed as `model__<param>` because the model is a step called `model` inside the pipeline.
`class_weight` is included in the search so the models can give more importance to the rare "churned" class.

In [ ]:
search_setup = {
    "Logistic Regression": GridSearchCV(
        build_pipeline(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        param_grid={
            "model__C": [0.01, 0.1, 1, 10, 100],
            "model__class_weight": [None, "balanced"],
        },
        cv=5, scoring="f1", n_jobs=-1,
    ),
    "KNN": GridSearchCV(
        build_pipeline(KNeighborsClassifier()),
        param_grid={
            "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2],
        },
        cv=5, scoring="f1", n_jobs=-1,
    ),
    "Decision Tree": GridSearchCV(
        build_pipeline(DecisionTreeClassifier(random_state=RANDOM_STATE)),
        param_grid={
            "model__max_depth": [3, 4, 5, 6, 8, 10, None],
            "model__min_samples_leaf": [1, 5, 10, 20, 50],
            "model__criterion": ["gini", "entropy"],
            "model__class_weight": [None, "balanced"],
        },
        cv=5, scoring="f1", n_jobs=-1,
    ),
    "Random Forest": RandomizedSearchCV(
        build_pipeline(RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        param_distributions={
            "model__n_estimators": [100, 200, 300, 400],
            "model__max_depth": [5, 8, 10, 12, 15, None],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__min_samples_split": [2, 5, 10],
            "model__max_features": ["sqrt", "log2", 0.5],
            "model__class_weight": [None, "balanced", "balanced_subsample"],
        },
        n_iter=25, cv=5, scoring="f1", n_jobs=1, random_state=RANDOM_STATE,
    ),
}

tuned = {}
for name, search in search_setup.items():
    start = time.time()
    search.fit(X_train, y_train)
    tuned[name] = search
    print(f"{name:20s} best CV F1 = {search.best_score_:.4f}  ({time.time() - start:.0f}s)")
    print("   best params:", search.best_params_)

## 7. Evaluate tuned models on the test set

In [ ]:
tuned_results = {}
for name, search in tuned.items():
    scores = evaluate(search.best_estimator_, X_test, y_test)
    scores["CV F1 (5-fold)"] = search.best_score_
    tuned_results[name] = scores

tuned_df = pd.DataFrame(tuned_results).T.round(4)
tuned_df

In [ ]:
# Baseline vs tuned (F1 and ROC-AUC on the test set)
compare = pd.DataFrame({
    "Baseline F1": baseline_df["F1"],
    "Tuned F1": tuned_df["F1"],
    "Baseline ROC-AUC": baseline_df["ROC-AUC"],
    "Tuned ROC-AUC": tuned_df["ROC-AUC"],
})
display_cols = compare.round(4)
print(display_cols)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
compare[["Baseline F1", "Tuned F1"]].plot.bar(ax=axes[0], rot=20)
axes[0].set_title("F1 score: baseline vs tuned")
axes[0].set_ylim(0, 1)
compare[["Baseline ROC-AUC", "Tuned ROC-AUC"]].plot.bar(ax=axes[1], rot=20)
axes[1].set_title("ROC-AUC: baseline vs tuned")
axes[1].set_ylim(0.5, 1)
plt.tight_layout()
plt.show()

## 8. Choose the best model

To avoid "peeking" at the test set, we select the winner using the **cross-validated F1 on the training data**; the test set is then used only to report the final, unbiased performance.

In [ ]:
best_name = max(tuned, key=lambda n: tuned[n].best_score_)
best_search = tuned[best_name]
best_pipeline = best_search.best_estimator_

print("Best model :", best_name)
print("Best params:", best_search.best_params_)
print("CV F1      : {:.4f}".format(best_search.best_score_))
print()
print("Test set performance:")
for metric, value in evaluate(best_pipeline, X_test, y_test).items():
    print(f"  {metric:10s}: {value:.4f}")

### 8.1 Confusion matrix, classification report and ROC curves

In [ ]:
y_pred = best_pipeline.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=["Stayed", "Left"],
                                        cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion matrix — {best_name}")

for name, search in tuned.items():
    RocCurveDisplay.from_estimator(search.best_estimator_, X_test, y_test, name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[1].set_title("ROC curves of tuned models")
axes[1].legend()
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=["Stayed", "Left"]))

### 8.2 Which features matter most?

We use **permutation importance**, which works for *any* model (including KNN and Logistic Regression): shuffle one original column at a time and see how much the F1 score drops.

In [ ]:
perm = permutation_importance(best_pipeline, X_test, y_test, scoring="f1",
                              n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)

importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values()
print(importance.sort_values(ascending=False).round(4))

plt.figure(figsize=(8, 5))
importance.plot.barh(color="#4c72b0")
plt.title(f"Permutation importance (drop in F1) — {best_name}")
plt.xlabel("Mean decrease in F1")
plt.tight_layout()
plt.show()

This should agree with the EDA: `Age`, `NumOfProducts`, `IsActiveMember`, `Geography` and `Balance` are the important features, while `HasCrCard`, `Tenure` and `EstimatedSalary` add very little.

## 9. Save the final model

We save the **entire pipeline** (preprocessing + model) with `joblib`, so new raw customer data can be passed in directly.
We also save:
- `metrics.json` — the results shown in the Streamlit app
- `test_data.csv` — the held-out test set, used in the prediction notebook

In [ ]:
joblib.dump(best_pipeline, "churn_model.pkl")

def clean_params(params):
    return {k.replace("model__", ""): (v if isinstance(v, (int, float, str, type(None))) else str(v))
            for k, v in params.items()}

cm = confusion_matrix(y_test, y_pred)

metrics = {
    "best_model": best_name,
    "best_params": clean_params(best_search.best_params_),
    "cv_f1": float(best_search.best_score_),
    "test_metrics": {k: float(v) for k, v in evaluate(best_pipeline, X_test, y_test).items()},
    "confusion_matrix": cm.tolist(),
    "baseline_results": baseline_df.reset_index().rename(columns={"index": "Model"}).to_dict(orient="records"),
    "tuned_results": tuned_df.reset_index().rename(columns={"index": "Model"}).to_dict(orient="records"),
    "feature_importance": {k: float(v) for k, v in importance.sort_values(ascending=False).items()},
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "sklearn_version": sklearn.__version__,
}
with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

test_data = X_test.copy()
test_data["Exited"] = y_test
test_data.to_csv("test_data.csv", index=False)

print("Saved: churn_model.pkl, metrics.json, test_data.csv")

In [ ]:
# Sanity check: reload the model and make sure it gives identical predictions
loaded = joblib.load("churn_model.pkl")
print("Predictions identical after reload:",
      np.array_equal(loaded.predict(X_test), best_pipeline.predict(X_test)))

## 10. Conclusion

- The tuned models were compared with cross-validation and the best one was chosen by cross-validated F1.
- The final pipeline was evaluated once on the untouched test set and saved as `churn_model.pkl`.
- **Next:** open `03_Predictions.ipynb` to use the saved model, and run `streamlit run app.py` for the web app.

**Ideas to improve further** (beyond what we covered): try SMOTE or threshold tuning to boost recall, engineer features such as `Balance/EstimatedSalary` or age groups, or try Gradient Boosting / XGBoost.